In [0]:
raw_path = "abfss://techdados@deadatabricks.dfs.core.windows.net/raw/customer/"
checkpoint_path = "abfss://techdados@deadatabricks.dfs.core.windows.net/tables/checkpoints"
schema_path = "abfss://techdados@deadatabricks.dfs.core.windows.net/tables/schema"

In [0]:
df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(raw_path)
)

# Escrever na bronze layer
(df.writeStream
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .table("techdados.bronze.clientes_autoloader")
)

In [0]:
# Verificar dados
spark.sql("SELECT * FROM techdados.bronze.clientes_autoloader").display()

In [0]:
# Verifica Schema
spark.sql("SELECT * FROM techdados.bronze.clientes_autoloader").printSchema()

In [0]:
#Exemplo de adição de colunas com novo arquivo
df_evolution = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") #Adicionado método para adição de colunas
    .load(raw_path)
)

(df_evolution.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")  # Permite adicionar colunas na tabela Delta
    .outputMode("append")
    .trigger(availableNow=True)
    .table("techdados.bronze.clientes_autoloader")
)

In [0]:
# Verificar dados
spark.sql("SELECT * FROM techdados.bronze.clientes_autoloader").display()

In [0]:
#Exemplo de rescue
df_rescue = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path + "/rescue")
    .option("cloudFiles.inferSchema", "true")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .load(raw_path)
)

(df_rescue.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path + "/rescue")
    .outputMode("append")
    .trigger(availableNow=True)
    .table("techdados.bronze.clientes_autoloader_rescue")
)

In [0]:
# Verificar dados
spark.sql("SELECT * FROM techdados.bronze.clientes_autoloader_rescue").display()

In [0]:
#Exemplo de trigger de tempo
df_rescue = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path + "/rescue")
    .option("cloudFiles.inferSchema", "true")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .load(raw_path)
)

(df_rescue.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path + "/rescue")
    .outputMode("append")
    .trigger(processingTime='15 seconds')
    .table("techdados.bronze.clientes_autoloader_rescue")
)

In [0]:
%sql
select * from techdados.bronze.clientes_autoloader_rescue